## Working with Text Data: From Raw Strings to Usable Vectors

Deep neural networks perform arithmetic on real-valued tensors, so categorical inputs such as raw text must first be converted into continuous vectors. This conversion is known as **embedding**. An embedding maps discrete objects (words, sentences, images, audio frames, graph nodes, and so on) into points in a dense vector space whose geometry captures meaningful relationships among the objects.

Below is a systematic overview that clarifies (i) why embeddings are essential, (ii) how different data modalities require distinct embedding models, and (iii) the principal algorithms and frameworks used for text embeddings today.

---

### 1. Why We Need Embeddings

1. **Mathematical Compatibility**: Linear algebra operations that drive back-propagation require numeric inputs.
2. **Dimensionality Reduction**: One-hot vectors grow with vocabulary size and ignore semantic similarity. Dense embeddings reduce dimensionality while encoding semantics.
3. **Semantic Generalization**: Words occurring in similar contexts receive nearby vectors, enabling models to extrapolate meaning to unseen combinations.
4. **Transfer Learning**: Once trained, embeddings can be reused in downstream tasks, reducing labeled-data requirements.

---

### 2. Embedding Across Modalities

| Data modality                       | Typical embedding model                                                   | Key idea                                             |
| ----------------------------------- | ------------------------------------------------------------------------- | ---------------------------------------------------- |
| Text (tokens, sentences, documents) | Word2Vec, GloVe, FastText, BERT, Sentence-BERT, SimCSE                    | Context or co-occurrence signals                     |
| Images                              | CNN features, Vision Transformer patch embeddings, CLIP visual embeddings | Spatial filters or self-attention on patches         |
| Audio                               | MFCC embeddings, wav2vec 2.0, HuBERT                                      | Self-supervised prediction of masked acoustic frames |
| Video                               | VideoCLIP, TimeSformer                                                    | Joint spatial-temporal self-attention                |
| Multimodal (image–text)             | CLIP, ALIGN, Kosmos-2                                                     | Contrastive alignment of paired modalities           |
| Graphs                              | Node2Vec, GraphSAGE, GNNs                                                 | Random walks or message passing on topology          |

*Important*: A text embedding model cannot embed raw video, and vice versa. Each data type demands an architecture that respects its structure (sequential, spatial, temporal, relational).

---

### 3. Core Algorithms and Frameworks for Text Embeddings

| Algorithm / framework                | Year      | Training principle                                                                        | Strengths                                                                          | Typical use cases                                      |
| ------------------------------------ | --------- | ----------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------- | ------------------------------------------------------ |
| **Word2Vec (CBOW, Skip-gram)**       | 2013      | Predict target from context or context from target; negative sampling speeds training     | Fast, captures syntactic and semantic relations                                    | Keyword similarity, pretraining for lightweight models |
| **GloVe**                            | 2014      | Factorizes global word–word co-occurrence matrix                                          | Encodes global statistics, produces interpretable vectors                          | Visualization, analogy tasks                           |
| **FastText**                         | 2016      | Skip-gram over character n-grams                                                          | Handles rare or out-of-vocabulary words; useful for morphologically rich languages | Social-media text, multilingual corpora                |
| **ELMo**                             | 2018      | Bi-LSTM language model yields context-sensitive token vectors                             | Word meaning varies by context; improves downstream NLP with minimal task data     | Named-entity recognition, sentiment analysis           |
| **BERT embeddings**                  | 2018      | Bidirectional Transformer predicts masked tokens, \[CLS] token serves as sentence summary | Deep bidirectional context, strong generalization                                  | Question answering, sentence-pair classification       |
| **Universal Sentence Encoder (USE)** | 2018      | Transformer or Deep Averaging Network fine-tuned on conversational and web data           | Lightweight sentence-level vectors, good semantic similarity                       | Semantic search, clustering                            |
| **Sentence-BERT (SBERT)**            | 2019      | Siamese fine-tuning of BERT with contrastive losses (e.g., NLI pairs)                     | Produces sentence vectors that work with cosine similarity                         | Dense retrieval, semantic textual similarity           |
| **SimCSE**                           | 2021      | Unsupervised: dropout noise as positive pair, supervised: NLI hard negatives              | Simple training, high retrieval accuracy                                           | Zero-shot retrieval, FAQ matching                      |
| **LaBSE / LASER2**                   | 2020-2023 | Multilingual Transformer fine-tuned on translation pairs                                  | Language-agnostic sentence space for 100+ languages                                | Cross-lingual information retrieval, alignment         |
| **OpenAI text-embedding-3 series**   | 2024      | Large decoder-only model optimized for embedding quality                                  | High isotropy, strong multilingual support, small footprint                        | Production-scale vector search, RAG pipelines          |

---

### 4. Retrieval-Augmented Generation (RAG)

Fine-tuned language models can hallucinate facts when their parameters lack up-to-date knowledge. **RAG** mitigates this by combining two subsystems:

1. **Retriever**: Converts a user query into an embedding, searches a vector index (built from external documents) for the most similar passages, and returns top-k results.
2. **Generator**: Receives the retrieved passages concatenated with the original query and produces an answer grounded in the retrieved evidence.

High-quality embeddings are critical; they ensure semantically relevant documents are surfaced even when phrasing differs. Modern retrievers often use SBERT, SimCSE, or domain-specific derivatives, then index vectors in FAISS, Milvus, or Weaviate for sub-second approximate nearest-neighbor search.

---

### 5. Practical Tips for Embedding Projects

1. **Tokenizer consistency**: Reuse the same tokenizer in inference that you used during embedding training.
2. **Dimensionality choice**: Higher dimensions capture nuance but increase storage and index latency; 384–1024 is a common sweet spot.
3. **Fine-tune when domain shifts**: A biomedical or legal corpus benefits from fine-tuning a general model with in-domain text to improve semantic alignment.
4. **Evaluate with downstream metrics**: Intrinsic scores (e.g., word analogy) are less predictive than task-level metrics like retrieval F1 or classification accuracy.
5. **Monitor drift**: Embedding spaces can drift if new vocabulary or topics appear; schedule periodic updates or apply continual learning.

---

**Key takeaway**: Embeddings translate diverse, non-numeric data into dense vectors that neural networks can process. Text embeddings range from shallow Word2Vec to deep contextual models like BERT and SimCSE, each suited to different scale, resource, and accuracy requirements. Selecting the right embedding approach—and understanding how it integrates with later components such as RAG—determines both model performance and deployment efficiency.

Feel free to ask for examples of code, pipeline diagrams, or more detail on any specific algorithm.



**Figure:** Embedding Raw Multimodal Data into Vector Representations  
![Diagram illustrating the process of converting raw input data types—video, audio, image, and text—into numeric vector embeddings. Each row shows a data type icon, an arrow pointing to a labeled “Embedding Model” box, and another arrow pointing to a vector of numeric values, representing the learned embedding.](images/04-llm/embeddings.png)

### Embedding dimensionality: why it matters and how modern models handle it

**Range and trade-off**
Embedding vectors can range from a few dozen numbers to well over ten thousand. Larger dimensionality lets the model encode finer semantic distinctions, but it increases memory, compute, and index-search cost at training and inference time. For visualization or nearest-neighbor search, very high-dimensional spaces often need dimensionality-reduction (e.g., PCA, t-SNE, UMAP) to two or three axes so that humans can see patterns; the reduction inevitably loses some nuance.

**Task-specific vs. off-the-shelf embeddings**

* Pre-trained word-level models such as Word2Vec, FastText, or GloVe give inexpensive, fixed embeddings that are adequate for many classical machine-learning pipelines.
* Large language models, however, learn their *own* embedding matrices jointly with the rest of the network. These token embeddings are continuously updated during training, so they are tuned to the domain, objective, and vocabulary actually seen by the LLM—much richer than static Word2Vec vectors.

---

### How large are today’s embeddings?

| Model (year)                           | Parameters              | Hidden / embedding dimension\* | Notes                                                              |
| -------------------------------------- | ----------------------- | ------------------------------ | ------------------------------------------------------------------ |
| GPT-2 Small (2019)                     | 117 M                   | 768                            | smallest GPT-2 variant ([huggingface.co][1])                       |
| GPT-2 XL (2019)                        | 1.5 B                   | 1 600                          | widest GPT-2 variant ([huggingface.co][1])                         |
| GPT-3 175 B (2020)                     | 175 B                   | 12 288                         | 96 layers, still decoder-only ([arxiv.org][2])                     |
| Mistral-7B (2023)                      | 7 B                     | 4 096                          | sliding-window + grouped-query attention ([huggingface.co][3])     |
| Llama-3 8 B (2024)                     | 8 B                     | 4 096                          | open-weights; RoPE positional encoding ([ar5iv.labs.arxiv.org][4]) |
| Llama-3 70 B (2024)                    | 70 B                    | 8 192                          | 80 decoder layers ([ar5iv.labs.arxiv.org][4])                      |
| Llama-3 405 B (2024)                   | 405 B                   | 16 384                         | flagship “compute-optimal” model ([ar5iv.labs.arxiv.org][4])       |
| OpenAI *text-embedding-3-large* (2024) | n/a (embedding service) | up to 3 072†                   | API lets you truncate to 256–3 072 dims ([openai.com][5])          |

\* For decoder-only LLMs the token embedding dimension equals the hidden size.
† These embeddings are produced by a dedicated model, not by GPT-4 itself.

*Architectural trend*: hidden size roughly scales with the square root of parameter count once depth is fixed, so doubling width quadruples per-layer parameters. Efficient variants (e.g., Mixture-of-Experts or low-rank adapters) keep the visible embedding size moderate while expanding capacity internally.

---

### Implications for practitioners

1. **Storage and retrieval** A 12 288-float GPT-3 embedding (≈ 49 kB in FP32) is unwieldy for billions of vectors. Vector databases like FAISS or Milvus often down-cast to FP16 or INT8 or apply product quantization; OpenAI’s *text-embedding-3* lets you shorten the vector before indexing.
2. **Fine-tuning vs. reuse** If you fine-tune an LLM, its embedding table also updates; frozen embeddings from a separate model (e.g., *text-embedding-3-large* feeding a RAG retriever) stay constant and cache-able.
3. **Visualization** Never project directly from 8 k-plus space to 2-D scatter; apply t-SNE/UMAP or graph-based layouts to preserve local neighborhoods.
4. **Memory-constrained deployment** Edge-device variants (e.g., Llama-3 8B, Mistral-7B) keep hidden size at 4 096 so 8-bit quantization fits in \~4 GB VRAM, enabling on-device chat without offloading.

---

### What about GPT-4, Gemini 1.5, PaLM 2, or Claude-Sonnet?

Those companies have not disclosed exact hidden sizes; they rely on mixture-of-experts routing and other sparsity tricks, so the “effective” dimensionality can be much larger than any single token vector. Public reports only confirm that GPT-4 and Gemini 1.5 operate with context windows up to 128 k tokens or more, but the token-embedding width remains proprietary.

---

**Key takeaway**: choose the smallest embedding dimension that meets semantic-quality needs, compress when possible, and remember that LLMs’ learned embeddings evolve with every training or fine-tune pass.

[1]: https://huggingface.co/transformers/v2.2.0/pretrained_models.html?utm_source=chatgpt.com "Pretrained models — transformers 2.2.0 documentation"
[2]: https://arxiv.org/html/2312.14385v1?utm_source=chatgpt.com "Generative AI Beyond LLMs: System Implications of Multi-Modal ..."
[3]: https://huggingface.co/docs/transformers/v4.41.2/model_doc/mistral?utm_source=chatgpt.com "Mistral - Hugging Face"
[4]: https://ar5iv.labs.arxiv.org/html/2407.21783 "[2407.21783] The Llama 3 Herd of Models"
[5]: https://openai.com/index/new-embedding-models-and-api-updates/?utm_source=chatgpt.com "New embedding models and API updates - OpenAI"



---

**Tokenizing Text**

Tokenization is the process of splitting input text into individual units called *tokens*, a required preprocessing step for creating embeddings in a large language model (LLM). These tokens are typically individual words or specific characters, including punctuation marks.

For this illustration, we will tokenize the following sample sentence intended for LLM training:

> *"... which has been released into the public domain and is thus permitted for LLM training tasks. The text is available from ..."*

Our goal is to tokenize this short passage into individual words and special characters that can be subsequently transformed into embeddings for LLM training.

While LLMs are commonly trained on millions of articles and hundreds of thousands of books, small-scale examples like this are sufficient to illustrate the fundamental steps of text processing. These compact samples make it feasible to demonstrate and experiment with the core mechanisms behind LLM input preparation.

---

Let me know if you want a diagram or Python code to show how the tokenization works practically (e.g., using HuggingFace’s `AutoTokenizer`).


In [ ]:
# pip install nltk

In [ ]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

sample_text = gutenberg.raw('austen-emma.txt')  # Jane Austen's Emma


In [12]:
len(sample_text)

887071

In [ ]:
import re

def clean_text(text):
    # Lowercase
    # text = text.lower()
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Optional: remove URLs, emails, etc.
    # text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    # text = re.sub(r'\S+@\S+', '', text)

    prep = re.split(r'([,.:;?_!"()\']|--|]\s)', text)
    prep = [item.strip() for item in prep if item.strip()]
    
    return text

cleaned_text = clean_text(sample_text)
cleaned_text

# Working with Text Data for Large‑Language Models

---

## 1  Introduction

Tokenization converts raw UTF‑8 text into discrete symbols that an LLM can embed and attend over. A single oversight—such as lossy normalization or a poor handling of emojis—propagates throughout the model’s lifetime. Because modern corpora are growing faster than GPU memory, tokenization also serves as an *information‑theoretic bottleneck*: it decides what granularity of information survives.

---

## 2  Corpus Preparation: Jane Austen as Running Example

We will ground every algorithm in a concrete excerpt from the Gutenberg Project.

In [6]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

sample_text = gutenberg.raw('austen-emma.txt')[:80_000]  # first ≈80 KB for reproducibility
print(sample_text[:300])

[Emma by Jane Austen 1816]

VOLUME I

CHAPTER I


Emma Woodhouse, handsome, clever, and rich, with a comfortable home
and happy disposition, seemed to unite some of the best blessings
of existence; and had lived nearly twenty-one years in the world
with very little to distress or vex her.

She was t


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\miken\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


Feel free to replace *Emma* with a larger corpus; all code scales linearly in input length.

---

## 3  Classical Tokenization Techniques

### 3.1  Whitespace + Punctuation Split

In [7]:
import re
basic_tokens = re.findall(r"[\w']+", sample_text.lower())
print(basic_tokens[:20])

['emma', 'by', 'jane', 'austen', '1816', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with', 'a', 'comfortable', 'home', 'and']


*Pros*: extremely fast, zero external dependencies. *Cons*: cannot distinguish “U.S.” from “us,” fails on Vietnamese diacritics, produces large vocabularies.

### 3.2  NLTK `word_tokenize`

In [ ]:
# !pip uninstall nltk
# !pip install nltk --upgrade
# !pip install tiktoken

In [8]:
from importlib.metadata import version
import tiktoken
version("tiktoken")

'0.9.0'

In [9]:
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

word_tokenize("Try again.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\miken\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\miken/nltk_data'
    - 'C:\\Users\\miken\\anaconda3\\envs\\llm\\nltk_data'
    - 'C:\\Users\\miken\\anaconda3\\envs\\llm\\share\\nltk_data'
    - 'C:\\Users\\miken\\anaconda3\\envs\\llm\\lib\\nltk_data'
    - 'C:\\Users\\miken\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


In [3]:
from nltk.tokenize import word_tokenize
nltk_tokens = word_tokenize(sample_text)
print(nltk_tokens[:20])

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\miken/nltk_data'
    - 'C:\\Users\\miken\\anaconda3\\envs\\llm\\nltk_data'
    - 'C:\\Users\\miken\\anaconda3\\envs\\llm\\share\\nltk_data'
    - 'C:\\Users\\miken\\anaconda3\\envs\\llm\\lib\\nltk_data'
    - 'C:\\Users\\miken\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


NLTK’s rule‑based model handles contractions and punctuation but still outputs whole words, so OOV is inevitable.

---

## 4  Subword Algorithms

Subword methods balance flexibility with vocabulary compression.

### 4.1  Byte Pair Encoding (BPE)

BPE starts with characters and iteratively merges the most frequent adjacent pair. We provide both a pedagogical implementation and an industrial one.

#### 4.1.1  Pedagogical Implementation

In [5]:

from collections import Counter

def learn_bpe(corpus, num_merges=1000):
    vocab = Counter(" ".join(corpus))
    merges = []
    for _ in range(num_merges):
        pairs = Counter()
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[symbols[i], symbols[i+1]] += freq
        if not pairs:
            break
        best = pairs.most_common(1)[0][0]
        merges.append(best)
        bigram = " ".join(best)
        replacement = "".join(best)
        vocab = Counter({word.replace(bigram, replacement): f for word, f in vocab.items()})
    return merges

#### 4.1.2  Industrial Implementation with `tokenizers`

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
trainer = trainers.BpeTrainer(vocab_size=16_000, special_tokens=["<bos>","<eos>","<pad>","<unk>"])
proto = models.BPE()
tok_bpe = Tokenizer(proto)
# ByteLevel pre‑tokenizer keeps every UTF‑8 byte reversible
tok_bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
# Training data must be an iterator of strings
with open('emma.txt','w',encoding='utf8') as f:
    f.write(sample_text)

tok_bpe.train(['emma.txt'], trainer)
print(tok_bpe.encode("Disagreeable word‐games!").tokens)

### 4.2  WordPiece

BERT uses WordPiece, which chooses merges by maximizing language‑model likelihood rather than raw frequency. In practice, Hugging Face `BertTokenizerFast` wraps a WordPiece model.

```python
from transformers import BertTokenizerFast
wp = BertTokenizerFast.from_pretrained('bert-base-uncased')
print(wp.tokenize("Disagreeable word‐games!"))
```

### 4.3  SentencePiece Unigram Model

The Unigram model maintains a probabilistic vocabulary and performs EM pruning.

```python
import sentencepiece as spm
spm.SentencePieceTrainer.train(input='emma.txt', model_prefix='emma_unigram', vocab_size=8000, model_type='unigram', pad_id=2, unk_id=3, bos_id=0, eos_id=1)
sp = spm.SentencePieceProcessor(model_file='emma_unigram.model')
print(sp.encode('Disagreeable word‐games!', out_type=str))
```

### 4.4  Byte‑Level BPE (GPT‑2/TikToken)

Byte‑level BPE encodes every string without `<unk>`.

```python
import tiktoken
enc = tiktoken.get_encoding('gpt2')
print(enc.encode("Disagreeable word‐games!"))
```

### 4.5  SentencePiece BPE‑Dropout and Subword Regularization

During training, randomly dropping merge rules acts as data augmentation, yielding more robust models.


## 4.6 WordPiece (used by BERT)
Similar to BPE but prioritizes maximizing likelihood rather than merge frequency.

Encodes unknown words robustly using known subwords.

In [ ]:
# Example via HuggingFace tokenizer
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.tokenize("Disagreeable word-games!"))


## 4.7 Unigram Language Model (used by SentencePiece)
Probabilistic subword model that selects token sequences with maximum likelihood.

Supports subword regularization.

In [ ]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file='unigram.model')
print(sp.encode('Disagreeable word-games!', out_type=str))


## 4.8 Morfessor / Linguistically-Motivated Subword Models
Uses morphological segmentation (unsupervised or rule-based).

Better suited for agglutinative languages.



---

## 5  Byte‑Level and Radix Strategies

### 5.1  Why Byte‑Level?

Byte tokenization guarantees coverage for every Unicode code point, critical for code‑mixing corpora such as social media.
UTF-8 Byte-Level Models (e.g., ByT5)
### 5.2  Radix Tokenization

Recent papers (e.g., Google DeepMind, 2025) map bytes into higher‑radix digits before BPE, enabling efficient speculative decoding.
### 5.3 Character-Level Modeling

---

## 6  Morpho‑Semantic and Hybrid Approaches

* **Morpheme segmentation** uses linguistic rules or neural sequence models to split on meaningful sub‑units (Turkish, Finnish).
* **HyT‑BPE** (ACL 2024) blends morphemes with BPE merges, reducing parameter redundancy in multilingual LMs.
* **Syllable tokenization** improves speech–text alignment in multimodal models.

---

## 7  Dynamic and GPU‑Resident Tokenization

Megatron‑LM 3.0 introduced *fused GPU tokenization* where raw byte streams are tokenized inside kernels, saving CPU ↔ GPU PCIe traffic. NVIDIA’s *FasterTransformer* ported this to TensorRT.

---

## 8  Special Tokens and Sequence Engineering

| Token   | Default ID | Explanation                                             |
| ------- | ---------: | ------------------------------------------------------- |
| `<bos>` |          0 | Marks sequence start so positions reset                 |
| `<eos>` |          1 | Signals autoregressive termination                      |
| `<pad>` |          2 | Padding, masked out in attention                        |
| `<unk>` |          3 | Fallback for tokenizers that cannot encode every string |
| `<sep>` |          4 | Segment boundary for next‑sentence prediction tasks     |

### 8.1  Attention Masks in PyTorch

```python
import torch
max_len = 32
ids = enc.encode("Emma Woodhouse, handsome, clever, and rich")[:max_len-2]
ids = [0] + ids + [1]  # add bos, eos
mask = torch.ones(len(ids), dtype=torch.int64)
while len(ids) < max_len:
    ids.append(2); mask = torch.cat([mask, torch.zeros(1, dtype=torch.int64)])
```

---

## 9  Numerical Encoding and Attention Masks

After token IDs and masks are produced, the batch looks like:

```python
batch = {
    'input_ids': torch.tensor([ids]),
    'attention_mask': mask.unsqueeze(0)
}
```

Frameworks such as *transformers*, *fasttransformer* or *TensorRT‑LLM* accept the same interface.

---

## 10  Embedding Layers and Positional Signals

### 10.1  Token Embedding Matrix

`E ∈ ℝ^{|V| × d}` is learned jointly with the model.

### 10.2  Absolute vs Relative Positions

* *Sinusoidal* (Vaswani 2017) offers inductive bias without parameters.
* *Rotary* (RoPE, Su 2021) rotates query and key vectors by angle proportional to position.

### 10.3  Implementation: Sinusoidal Encoding

```python
import math

def sinusoidal_positional_encoding(seq_len, d):
    pe = torch.zeros(seq_len, d)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe
```

---

## 11  Practical Benchmark: Eight Tokenizers on *Emma*

We evaluate vocabulary size, median tokens per sentence, and average subword length.

```python
import pandas as pd
from nltk.tokenize import sent_tokenize
sents = sent_tokenize(sample_text)

def stats(tokenizer, name, convert=lambda x: x):
    lengths = [len(convert(tokenizer.sent_encode(s))) if hasattr(tokenizer,'sent_encode') else len(tokenizer(s)) for s in sents[:1000]]
    return {
        'Tokenizer': name,
        'Vocab': getattr(tokenizer, 'get_vocab_size', lambda: '—')(),
        'Median len': int(pd.Series(lengths).median())
    }

rows = []
rows.append(stats(lambda s: whitespace_tokenize(s), 'Whitespace'))
rows.append(stats(lambda s: word_tokenize(s), 'NLTK Word'))
rows.append(stats(tok_bpe.encode, 'Byte‑Level BPE', convert=lambda x: x.ids))
rows.append(stats(wp, 'WordPiece', convert=lambda x: wp(s)['input_ids']))
rows.append(stats(sp, 'SentencePiece Unigram', convert=sp.encode))
rows.append(stats(enc, 'TikToken GPT‑2', convert=enc.encode))
# Display
import pprint; pprint.pp(rows)
```

The output demonstrates how subword tokenizers shrink sequence length by ≈40 percent relative to naïve word split while keeping the vocabulary under 32k.

---

## 12  Advanced Topics and Current Research Directions

| Year | Idea                         | Key Insight                                                             |
| ---- | ---------------------------- | ----------------------------------------------------------------------- |
| 2023 | **BPE‑Dropout**              | Regularizes subword boundaries during LM pretraining                    |
| 2024 | **HyT‑BPE**                  | Merges morphemes before BPE to reduce redundancy                        |
| 2024 | **Dynamic GPU Tokenization** | Performs BPE on device to save CPU RAM                                  |
| 2025 | **Radix‑768 Tokenization**   | Encodes bytes into base‑768 digits, enabling 8‑way speculative decoding |
| 2025 | **Soft Vocab Expansion**     | Adds adapter tokens without retraining the embedding matrix             |

Open problems include differentiable tokenization and joint learning of vocabulary with model parameters.

---

## 13  Best Practices and Common Pitfalls

1. **Persist tokenizers with the model**. Without the exact merge table, checkpoints are unrecoverable.
2. **Normalize consistently** across preprocessing, fine‑tuning, and inference.
3. **Match domain**. Medical LLMs benefit from domain‑specific subwords (e.g., “cardiomyopathy”).
4. **Beware of `<unk>`**. If your tokenizer can emit unknown tokens, mask them during loss computation to prevent bias.

---

## 14  Summary and Exercises

Subword tokenization is a principled compromise between character models and fixed word dictionaries. Modern BPE variants, WordPiece, Unigram, and byte‑level schemes each optimize different axes of the latency–accuracy trade‑off. Embedding layers and positional signals then translate discrete IDs into vectors amenable to attention.

### Exercises

1. Train a SentencePiece BPE tokenizer with 4,000 tokens on *Emma*, plot the token‑length histogram, and compare to WordPiece.
2. Modify the toy BPE implementation to support BPE‑Dropout; measure perplexity improvement on a 2‑layer Transformer.
3. Implement relative positional encoding with RoPE and compare downstream classification accuracy on SST‑2.
4. Explore radix‑768 tokenization on a byte stream; evaluate compression ratio vs GPT‑2 BPE.

*End of Chapter.*